## Topic: Document-Structure Based Text Splitter

### Agenda

- 1. Introduction of Structure-Aware Based Text Splitter

- 2. Taxonomy of Structure-Aware Splitters

- 3. Summary of Text Splitter


### 1. Introduction of Structure-Aware Based Text Splitter

- Definition:
    - Structure-Aware Text Splitters split documents based on their logical structure, instead of splitting at arbitrary character counts. They understand the document's format (Markdown, HTML, Code, JSON) and split at meaningful boundaries.

    

In [ ]:
"""  
        The Core Difference:

┌──────────────────────────────────────────────────────────────┐
│     CHARACTER-BASED vs STRUCTURE-AWARE                       │
│                                                              │
│  CHARACTER-BASED (RecursiveCharacterTextSplitter)            │
│  ────────────────────────────────────────────                │
│  "Give me chunks of ~1000 chars, break at nice spots."       │
│  Knows: paragraphs, sentences, words                         │
│  Doesn't know: WHAT the text is (a header? a table? a func?) │
│                                                              │
│  STRUCTURE-AWARE (Header/HTML/Code/JSON splitters)           │
│  ────────────────────────────────────────────                │
│  "Split where the AUTHOR split — at sections, tags, defs."   │
│  Knows: document hierarchy, element types, scope             │
│  Bonus: attaches structural metadata to every chunk          │
│                                                              │
│  Input:  "# Guide\n## Install\npip install x\n## Usage\n..." │
│                                                              │
│  Character-based output:                                     │
│    Chunk 1: "# Guide\n## Install\npip install x\n## Usa"     │
│    Chunk 2: "ge\n..."               ← cut mid-header         │
│                                                              │
│  Structure-aware output:                                     │
│    Chunk 1: "pip install x"                                  │
│             metadata={"H1": "Guide", "H2": "Install"}        │
│    Chunk 2: "..."                                            │
│             metadata={"H1": "Guide", "H2": "Usage"}          │
└──────────────────────────────────────────────────────────────┘

"""

### 2. Taxonomy of Structure-Aware Splitters

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│              STRUCTURE-AWARE SPLITTER TAXONOMY                   │
│                                                                  │
│  📝 MARKDOWN                                                     │
│  ├── MarkdownHeaderTextSplitter        → Split by #, ##, ###     │
│  ├── MarkdownTextSplitter              → Recursive w/ MD seps    │
│  └── ExperimentalMarkdownSyntaxTextSplitter → Code/HR aware      │
│                                                                  │
│  🌐 HTML                                                         │
│  ├── HTMLHeaderTextSplitter            → Split by <h1>–<h6>      │
│  ├── HTMLSectionSplitter               → Split into sections     │
│  └── HTMLSemanticPreservingSplitter    → Keeps tables/lists      │
│                                                                  │
│  💻 CODE                                                         │
│  ├── RecursiveCharacterTextSplitter.from_language()  ⭐          │
│  ├── Language enum (Python, JS, TS, Java, Go, Rust, C++, ...)    │
│  └── PythonCodeTextSplitter (legacy convenience class)           │
│                                                                  │
│  📦 OTHER STRUCTURED FORMATS                                     │
│  ├── RecursiveJsonSplitter             → Valid JSON sub-objects  │
│  ├── LatexTextSplitter / Language.LATEX → \section, \begin{}     │ 
│  ├── Language.RST                      → reStructuredText        │
│  └── Table-aware strategies (CSV/Excel rows, HTML tables)        │
└──────────────────────────────────────────────────────────────────┘

"""

In [ ]:
""" 
- The major categories: 

        Structure-Aware Splitters
        │
        ├── Markdown
        │   └── MarkdownHeaderTextSplitter
        │
        ├── HTML
        │   └── HTMLHeaderTextSplitter
        │
        ├── Code
        │   └── Language-aware splitters
        │
        ├── JSON
            └── JSON-aware splitting approaches


"""

In [ ]:
"""     - Decision Tree
        ==================

What format is your content?
│
├─ Markdown (.md, README, Notion export, Obsidian)
│   ├─ Need section breadcrumbs? ──► MarkdownHeaderTextSplitter + size stage 
│   └─ Just need size limits?   ──► MarkdownTextSplitter
│
├─ HTML (web pages, scraped docs)
│   ├─ Has tables/lists/code to protect? ──► HTMLSemanticPreservingSplitter
│   ├─ Want nested header metadata?      ──► HTMLHeaderTextSplitter + size stage 
│   └─ Want readable full sections?      ──► HTMLSectionSplitter
│
├─ Source code (.py, .js, .go, .java, ...)
│   └──► RecursiveCharacterTextSplitter.from_language(Language.X) ──► x= Python, Java
│        (overlap=0, add symbol/file metadata yourself)
│
├─ JSON (API exports, configs, catalogs)
│   └──► RecursiveJsonSplitter(convert_lists=True) 
│
├─ LaTeX / RST (papers, Sphinx docs)
│   └──► from_language(Language.LATEX / Language.RST)
│
├─ Tabular (CSV, Excel)
│   └──► CSVLoader (row = chunk) → group rows if tiny; repeat headers if wide
│
└─ Plain text / PDF text with no structure
    └──► RecursiveCharacterTextSplitter (previous deep dive)


"""

### 3. Summary of Text Splitter

In [ ]:
"""  
┌──────────────────────────────────────────────────────────────────┐
│               STRUCTURE-AWARE TEXT SPLITTERS                     │
│               ===============================                    │
│  WHAT:  Split by the format's own structure (headers, tags,      │
│         functions, keys) instead of raw character counts         │
│  WHY:   Coherent chunks + rich metadata + no broken atomic units │
│                                                                  │
│  📝 MARKDOWN                                                     │
│    MarkdownHeaderTextSplitter → {H1, H2, H3} metadata, no size   │
│    MarkdownTextSplitter       → size control, MD separators      │
│                                                                  │
│  🌐 HTML                                                         │
│    HTMLHeaderTextSplitter          → h1-h6 metadata, URL loading │
│    HTMLSectionSplitter             → readable sections           │
│    HTMLSemanticPreservingSplitter  → size + intact tables/lists  │
│                                                                  │
│  💻 CODE                                                         │
│    RecursiveCharacterTextSplitter.from_language(Language.X)      │
│    → splits at class/def/function; use chunk_overlap=0           │
│                                                                  │
│  📦 OTHER                                                        │
│    RecursiveJsonSplitter     → always-valid JSON sub-objects     │
│    Language.LATEX / .RST     → \section, environments, headers   │
│    CSV rows                  → loader already chunks per row     │
│                                                                  │
│  ⭐ THE PATTERN:                                                 │
│    Stage 1: structure splitter  → sections + metadata            │
│    Stage 2: size splitter       → .split_documents(sections)     │
│    Bonus  : prepend breadcrumb  → "H1 > H2 > H3\n\n<text>"       │
│                                                                  │
│  PIPELINE:                                                       │
│    [Loader] → [Structure Split] → [Size Split] → [Embed] → [VS]  │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Let the author's structure draw the first cut. Let chunk_size  │
│   draw the second. Carry the structure along as metadata."       │
└──────────────────────────────────────────────────────────────────┘

"""